# **Insta Sugar Intelligence**


Objective: Help Sugar Cosmetics’ marketing team understand which formats, campaigns, posting days and hashtags are associated with higher engagement, so they can plan better content.

In [43]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# **Step 2: Understanding the Data**

In [44]:
DATA_PATH = 'https://raw.githubusercontent.com/Vipsa2207/Insta-Sugar-Intelligence/main/data/sugar_cosmetics_instagram_data.xlsx'

In [45]:
df = pd.read_excel(DATA_PATH)

assert df.shape == (50000, 14), f"Unexpected shape: {df.shape}"
print("Loaded OK:", df.shape)

Loaded OK: (50000, 14)


In [46]:
display(df.head())
df.info()

,Post Date,Post Type,Campaign Type,Hashtags,Audience Emotion,Engagement Source,Impressions,Reach,Likes,Comments,Shares,Saves,Profile Visits,Follows
0,2024-12-23,Story,Giveaway,"#SugarCosmetics, #MakeupGoals, #BoldLooks",Curious,Collab,16795,15665,3076,430,73,99,193,86
1,2024-06-01,Image,Tutorial,"#BoldLooks, #MakeupGoals, #VeganBeauty",Confident,Influencer,45131,29335,5285,308,396,502,294,146
2,2024-04-28,Reel,Festive Offer,#VeganBeauty,Confident,Collab,18568,13118,2032,233,73,140,204,44
3,2025-02-01,Reel,Product Launch,"#CrueltyFree, #VeganBeauty, #SugarCosmetics",Happy,Collab,1189,751,130,13,12,7,33,12
4,2024-08-04,Story,Tutorial,"#SugarCosmetics, #VeganBeauty",Curious,Organic,15502,9373,2234,300,99,47,180,35


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Post Date          50000 non-null  datetime64[ns]
 1   Post Type          50000 non-null  object        
 2   Campaign Type      50000 non-null  object        
 3   Hashtags           50000 non-null  object        
 4   Audience Emotion   50000 non-null  object        
 5   Engagement Source  50000 non-null  object        
 6   Impressions        50000 non-null  int64         
 7   Reach              50000 non-null  int64         
 8   Likes              50000 non-null  int64         
 9   Comments           50000 non-null  int64         
 10  Shares             50000 non-null  int64         
 11  Saves              50000 non-null  int64         
 12  Profile Visits     50000 non-null  int64         
 13  Follows            50000 non-null  int64         
dtypes: dat

In [47]:
display(df.describe().T)

,count,mean,min,25%,50%,75%,max,std
Post Date,50000,2024-10-17 22:51:23.904000512,2024-04-19 00:00:00,2024-07-18 00:00:00,2024-10-18 00:00:00,2025-01-17 00:00:00,2025-04-19 00:00:00,NaN
Impressions,"50,000.00","25,520.92","1,002.00","13,243.00","25,541.00","37,758.25","49,999.00","14,118.65"
Reach,"50,000.00","19,748.47",616.00,"10,157.75","19,393.50","28,763.25","47,329.00","11,311.96"
Likes,"50,000.00","2,954.57",40.00,"1,256.00","2,466.00","4,255.00","11,392.00","2,144.92"
Comments,"50,000.00",369.19,2.00,135.00,282.00,516.00,"2,205.00",311.68
Shares,"50,000.00",162.31,0.00,50.00,114.00,228.00,"1,072.00",151.72
Saves,"50,000.00",177.23,1.00,62.00,132.00,246.00,"1,057.00",154.95
Profile Visits,"50,000.00",592.70,7.00,252.00,497.00,851.00,"2,264.00",428.22
Follows,"50,000.00",177.33,0.00,63.00,133.00,245.00,"1,114.00",154.55


In [48]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    print(f"\n{col}  ->  {df[col].nunique()} unique values")
    if df[col].nunique() <= 10:
        print(df[col].value_counts())


Post Type  ->  4 unique values
Post Type
Reel     12599
Story    12517
Video    12485
Image    12399
Name: count, dtype: int64

Campaign Type  ->  5 unique values
Campaign Type
Tutorial             10186
Giveaway              9998
Festive Offer         9990
Product Launch        9923
Influencer Collab     9903
Name: count, dtype: int64

Hashtags  ->  85 unique values

Audience Emotion  ->  5 unique values
Audience Emotion
Inspired     10137
Excited      10061
Curious      10027
Confident     9891
Happy         9884
Name: count, dtype: int64

Engagement Source  ->  4 unique values
Engagement Source
Collab        12633
Organic       12508
Influencer    12501
Paid          12358
Name: count, dtype: int64


In [49]:
print("First date:", df['Post Date'].min())
print("Last date: ", df['Post Date'].max())

print("\nRecords per month:")
print(df['Post Date'].dt.to_period('M').value_counts().sort_index())

First date: 2024-04-19 00:00:00
Last date:  2025-04-19 00:00:00

Records per month:
Post Date
2024-04    1626
2024-05    4250
2024-06    4232
2024-07    4121
2024-08    4242
2024-09    4160
2024-10    4225
2024-11    4075
2024-12    4296
2025-01    4244
2025-02    3930
2025-03    4139
2025-04    2460
Freq: M, Name: count, dtype: int64


## Data Dictionary

| Column | Type | Meaning |
|---|---|---|
| Post Date | Date | Date of the post |
| Post Type | Category | Image, Reel, Story or Video |
| Campaign Type | Category | Giveaway, Tutorial, Festive Offer, Product Launch or Influencer Collab |
| Hashtags | Text | Hashtags used, separated by commas |
| Audience Emotion | Category | Curious, Confident, Happy, Excited or Inspired |
| Engagement Source | Category | Organic, Paid, Collab or Influencer |
| Impressions | Number | Total views, including repeat views |
| Reach | Number | Unique accounts that saw the post |
| Likes | Number | Number of likes |
| Comments | Number | Number of comments |
| Shares | Number | Number of shares |
| Saves | Number | Number of saves |
| Profile Visits | Number | Visits to the brand profile after seeing the post |
| Follows | Number | New follows after seeing the post |

**Notes:** There is no Caption or Post ID column. Each row is treated as one post record.

Each row is one post record with its format, campaign, hashtags, emotion, source and performance numbers. It doesn’t tell us the post’s content, exact time, the follower count, or who engaged, so we can describe patterns but not explain why.

# **Step 3: Cleaning and Validation**

In [50]:
clean = df.copy()

In [51]:
print("Missing values per column:")
print(clean.isna().sum())
print("\nDuplicate rows:", clean.duplicated().sum())

Missing values per column:
Post Date            0
Post Type            0
Campaign Type        0
Hashtags             0
Audience Emotion     0
Engagement Source    0
Impressions          0
Reach                0
Likes                0
Comments             0
Shares               0
Saves                0
Profile Visits       0
Follows              0
dtype: int64

Duplicate rows: 0


In [52]:
text_cols = clean.select_dtypes(include='object').columns
for col in text_cols:
    clean[col] = clean[col].str.strip()

for col in ['Post Type', 'Campaign Type', 'Audience Emotion', 'Engagement Source']:
    print(col, '->', sorted(clean[col].unique()))

Post Type -> ['Image', 'Reel', 'Story', 'Video']
Campaign Type -> ['Festive Offer', 'Giveaway', 'Influencer Collab', 'Product Launch', 'Tutorial']
Audience Emotion -> ['Confident', 'Curious', 'Excited', 'Happy', 'Inspired']
Engagement Source -> ['Collab', 'Influencer', 'Organic', 'Paid']


In [53]:
interactions = clean[['Likes', 'Comments', 'Shares', 'Saves']].sum(axis=1)

checks = {
    "No negative numbers":        (clean.select_dtypes('number') >= 0).all().all(),
    "Reach <= Impressions":       (clean['Reach'] <= clean['Impressions']).all(),
    "Interactions <= Reach":      (interactions <= clean['Reach']).all(),
    "Profile Visits <= Reach":    (clean['Profile Visits'] <= clean['Reach']).all(),
    "Follows <= Profile Visits":  (clean['Follows'] <= clean['Profile Visits']).all(),
    "Dates in expected range":    clean['Post Date'].between(pd.Timestamp('2024-04-19'),
                                                             pd.Timestamp('2025-04-19')).all(),
}

report = pd.DataFrame({'Check': list(checks.keys()), 'Passed': list(checks.values())})
display(report)
assert report['Passed'].all(), "A validation check failed"

,Check,Passed
0,No negative numbers,True
1,Reach <= Impressions,True
2,Interactions <= Reach,True
3,Profile Visits <= Reach,True
4,Follows <= Profile Visits,True
5,Dates in expected range,True


In [54]:
count_cols = ['Impressions', 'Reach', 'Likes', 'Comments', 'Shares', 'Saves',
              'Profile Visits', 'Follows']

rows = []
for c in count_cols:
    q1, q3 = clean[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((clean[c] < low) | (clean[c] > high)).sum()
    rows.append({'Column': c, 'Outliers': n, 'Share %': round(n / len(clean) * 100, 2)})

display(pd.DataFrame(rows))

,Column,Outliers,Share %
0,Impressions,0,0.00
1,Reach,0,0.00
2,Likes,674,1.35
3,Comments,1836,3.67
4,Shares,2180,4.36
5,Saves,2101,4.20
6,Profile Visits,635,1.27
7,Follows,2152,4.30


No missing values, duplicates or invalid values found. Outliers are high-performing posts and were kept.

# **Step 4: Calculated Fields**

In [55]:
clean['Total Engagement'] = clean[['Likes', 'Comments', 'Shares', 'Saves']].sum(axis=1)
clean['Engagement Rate']  = clean['Total Engagement'] / clean['Reach'] * 100
clean['Profile Visit Rate'] = clean['Profile Visits'] / clean['Reach'] * 100
clean['Follow Conversion']  = clean['Follows'] / clean['Profile Visits'] * 100

In [56]:
clean['Year-Month'] = clean['Post Date'].dt.to_period('M').astype(str)
clean['Month Name'] = clean['Post Date'].dt.strftime('%b %Y')

weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
clean['Weekday'] = pd.Categorical(clean['Post Date'].dt.day_name(),
                                  categories=weekday_order, ordered=True)

clean['Is Partial Month'] = clean['Year-Month'].isin(['2024-04', '2025-04'])

In [57]:
clean['Hashtag Count'] = clean['Hashtags'].str.split(', ').str.len()

hashtag_df = clean.assign(Hashtag=clean['Hashtags'].str.split(', ')).explode('Hashtag')

print("Rows in clean:      ", len(clean))
print("Rows in hashtag_df: ", len(hashtag_df))
print("Unique hashtags:    ", hashtag_df['Hashtag'].nunique())
print(hashtag_df['Hashtag'].value_counts())

Rows in clean:       50000
Rows in hashtag_df:  100044
Unique hashtags:     5
Hashtag
#CrueltyFree       20197
#MakeupGoals       20018
#VeganBeauty       19982
#BoldLooks         19956
#SugarCosmetics    19891
Name: count, dtype: int64


In [58]:
print(clean.loc[0, ['Likes', 'Comments', 'Shares', 'Saves', 'Total Engagement', 'Reach', 'Engagement Rate']])

assert clean[['Engagement Rate', 'Profile Visit Rate', 'Follow Conversion']].notna().all().all()
assert np.isfinite(clean[['Engagement Rate', 'Profile Visit Rate', 'Follow Conversion']]).all().all()

display(clean[['Total Engagement', 'Engagement Rate', 'Profile Visit Rate', 'Follow Conversion']].describe().T)

Likes                3076
Comments              430
Shares                 73
Saves                  99
Total Engagement     3678
Reach               15665
Engagement Rate     23.48
Name: 0, dtype: object


,count,mean,std,min,25%,50%,75%,max
Total Engagement,"50,000.00","3,663.29","2,669.35",50.00,"1,552.00","3,058.50","5,271.00","14,980.00"
Engagement Rate,"50,000.00",18.54,7.20,5.59,12.32,18.49,24.64,34.10
Profile Visit Rate,"50,000.00",3.00,1.15,0.94,2.00,3.01,4.00,5.00
Follow Conversion,"50,000.00",29.78,11.59,0.00,19.70,29.72,39.87,49.96


# **Step 5: Visuals**

## Visual 1: KPI Overview

In [59]:
kpis = {
    'Total Posts':                   len(clean),
    'Total Impressions':             clean['Impressions'].sum(),
    'Total Reach (post-level)':      clean['Reach'].sum(),
    'Total Likes':                   clean['Likes'].sum(),
    'Total Comments':                clean['Comments'].sum(),
    'Total Shares':                  clean['Shares'].sum(),
    'Total Saves':                   clean['Saves'].sum(),
    'Total Engagement':              clean['Total Engagement'].sum(),
    'Avg Engagement Rate (%)':       clean['Engagement Rate'].mean(),
    'Median Engagement Rate (%)':    clean['Engagement Rate'].median(),
    'Avg Profile Visit Rate (%)':    clean['Profile Visit Rate'].mean(),
    'Avg Follow Conversion (%)':     clean['Follow Conversion'].mean(),
}

for name, value in kpis.items():
    text = f"{value:>20,.2f}" if isinstance(value, float) else f"{value:>20,}"
    print(f"{name:<30}{text}")

Total Posts                                 50,000
Total Impressions                    1,276,045,750
Total Reach (post-level)               987,423,288
Total Likes                            147,728,279
Total Comments                          18,459,361
Total Shares                             8,115,338
Total Saves                              8,861,439
Total Engagement                       183,164,417
Avg Engagement Rate (%)                      18.54
Median Engagement Rate (%)                   18.49
Avg Profile Visit Rate (%)                    3.00
Avg Follow Conversion (%)                    29.78


**Insight, So what and Action:** Across 50,000 post records, the content earned 1.276 billion impressions, 987.4 million post-level reach (the same person can be counted in many posts) and 183.2 million interactions, with an average engagement rate of 18.54% (median 18.49%). About 3% of reached accounts visit the profile and about 30% of those visitors follow. The so-what is that 18.5% is the baseline for judging any format, campaign or day, because a result is only good if it beats this number by more than luck. The action is to measure future content against this baseline and to track saves and shares separately, since likes are 80.7% of all interactions.

## Visual 2: Monthly Posting Trend

In [60]:
import plotly.express as px

monthly = (clean.groupby('Year-Month')
           .agg(Posts=('Post Date', 'size'),
                Days=('Post Date', lambda s: s.dt.date.nunique()),
                Partial=('Is Partial Month', 'first'))
           .reset_index())

monthly['Posts per Day'] = (monthly['Posts'] / monthly['Days']).round(1)
monthly['Month Type'] = np.where(monthly['Partial'], 'Partial month', 'Full month')

fig = px.bar(monthly, x='Year-Month', y='Posts', color='Month Type',
             hover_data=['Days', 'Posts per Day'],
             title='Monthly Post Volume (partial months highlighted)')
fig.update_xaxes(type='category', tickangle=-45)
fig.show()

display(monthly[['Year-Month', 'Posts', 'Days', 'Posts per Day']])

,Year-Month,Posts,Days,Posts per Day
0,2024-04,1626,12,135.50
1,2024-05,4250,31,137.10
2,2024-06,4232,30,141.10
3,2024-07,4121,31,132.90
4,2024-08,4242,31,136.80
5,2024-09,4160,30,138.70
6,2024-10,4225,31,136.30
7,2024-11,4075,30,135.80
8,2024-12,4296,31,138.60
9,2025-01,4244,31,136.90


**Insight, So what and Action:** Posting is steady at about 135 post records per day, between 130 and 141 in every month. The low counts for April 2024 (1,626) and April 2025 (2,460) are partial months with only 12 and 19 days of data, not real drops, so the chart marks them separately. The so-what is that posting volume does not change across the year, so the monthly count alone cannot explain changes in engagement. The action is to keep a steady posting rhythm, to plan content by format and campaign, and to compare months using posts per day instead of raw counts.

## Visuals 3 and 4: Top 10 Posts

In [61]:
clean['Post ID'] = 'P' + (clean.index + 1).astype(str).str.zfill(5)

clean['Post Label'] = (clean['Post ID'] + ' | '
                       + clean['Post Date'].dt.strftime('%d %b %Y') + ' | '
                       + clean['Post Type'] + ' | '
                       + clean['Campaign Type'])

In [62]:
top_likes = clean.nlargest(10, 'Likes')[['Post Label', 'Likes', 'Reach', 'Engagement Rate']]

fig3 = px.bar(top_likes.sort_values('Likes'), x='Likes', y='Post Label',
              orientation='h', hover_data=['Reach', 'Engagement Rate'],
              title='Top 10 Posts by Likes')
fig3.update_layout(yaxis_title=None)
fig3.show()
display(top_likes)

,Post Label,Likes,Reach,Engagement Rate
27397,P27398 | 26 Oct 2024 | Reel | Influencer Collab,11392,46789,31.61
29154,P29155 | 05 May 2024 | Image | Influencer Collab,11343,46049,31.55
4081,P04082 | 04 Mar 2025 | Video | Influencer Collab,11282,45271,30.35
34086,P34087 | 10 Apr 2025 | Image | Giveaway,11248,46124,30.06
16089,P16090 | 03 Nov 2024 | Story | Giveaway,11108,44730,29.02
16989,P16990 | 20 Jun 2024 | Story | Festive Offer,11083,46177,30.07
39991,P39992 | 22 Jun 2024 | Image | Product Launch,11060,44780,31.30
21289,P21290 | 17 May 2024 | Reel | Festive Offer,11052,44505,32.98
22782,P22783 | 25 Feb 2025 | Reel | Giveaway,11049,44343,30.48
9075,P09076 | 18 Oct 2024 | Video | Product Launch,11043,44307,29.61


In [63]:
top_er = clean.nlargest(10, 'Engagement Rate')[['Post Label', 'Engagement Rate', 'Reach', 'Likes']]

fig4 = px.bar(top_er.sort_values('Engagement Rate'), x='Engagement Rate', y='Post Label',
              orientation='h', hover_data=['Reach', 'Likes'],
              title='Top 10 Posts by Engagement Rate (%)')
fig4.update_layout(yaxis_title=None)
fig4.show()
display(top_er)

,Post Label,Engagement Rate,Reach,Likes
23784,P23785 | 05 Apr 2025 | Image | Influencer Collab,34.10,12710,3152
44717,P44718 | 08 Oct 2024 | Reel | Giveaway,34.01,38468,9510
4146,P04147 | 21 May 2024 | Story | Giveaway,34.01,16322,3996
1784,P01785 | 13 Mar 2025 | Story | Giveaway,33.98,27896,6961
48035,P48036 | 20 Oct 2024 | Video | Product Launch,33.96,7181,1791
34384,P34385 | 13 Nov 2024 | Image | Influencer Collab,33.88,25469,6324
23302,P23303 | 08 Feb 2025 | Reel | Tutorial,33.88,35746,8932
8367,P08368 | 11 Jan 2025 | Video | Festive Offer,33.87,10304,2564
34329,P34330 | 08 Feb 2025 | Video | Tutorial,33.86,21502,5345
5464,P05465 | 15 Nov 2024 | Video | Product Launch,33.82,44299,10844


In [64]:
overlap = set(top_likes['Post Label']) & set(top_er['Post Label'])
print("Posts in both Top 10 lists:", len(overlap))

print(clean[['Reach', 'Likes', 'Engagement Rate']].corr().round(2))

Posts in both Top 10 lists: 0
                 Reach  Likes  Engagement Rate
Reach             1.00   0.79             0.00
Likes             0.79   1.00             0.53
Engagement Rate   0.00   0.53             1.00


**Insight, So what and Action:** The 10 most-liked posts all had a very large reach (about 44,000 to 47,000 accounts), and none of them is in the top 10 by engagement rate. Likes and reach are strongly related (correlation 0.79), while reach and engagement rate are not (0.00). The top 10 by engagement rate sit between 33.8% and 34.1% and come from every format and campaign, with reach from about 7,000 to 44,000. The so-what is that likes mostly show which posts had the biggest audience, whereas engagement rate shows how well the content connected with the people who saw it, and the best posts do not point to one winning formula. The action is to judge content by engagement rate, not likes, and not to copy one "top" post, since 10 posts out of 50,000 are not a pattern.

## Visual 5: Post Type Comparison

In [65]:
format_summary = (clean.groupby('Post Type')
    .agg(Posts=('Post ID', 'count'),
         Avg_Likes=('Likes', 'mean'),
         Avg_Comments=('Comments', 'mean'),
         Avg_Shares=('Shares', 'mean'),
         Avg_Saves=('Saves', 'mean'),
         Avg_Engagement_Rate=('Engagement Rate', 'mean'),
         Median_Engagement_Rate=('Engagement Rate', 'median'))
    .round(2)
    .reset_index())

display(format_summary)

,Post Type,Posts,Avg_Likes,Avg_Comments,Avg_Shares,Avg_Saves,Avg_Engagement_Rate,Median_Engagement_Rate
0,Image,12399,"2,953.54",366.82,160.72,176.89,18.43,18.30
1,Reel,12599,"2,959.59",369.21,162.20,178.00,18.61,18.61
2,Story,12517,"2,964.24",370.65,164.62,177.58,18.60,18.55
3,Video,12485,"2,940.82",370.05,161.66,176.44,18.51,18.47


In [66]:
stats = clean.groupby('Post Type')['Engagement Rate'].agg(['mean', 'std', 'count']).reset_index()
stats['Margin'] = 1.96 * stats['std'] / np.sqrt(stats['count'])

fig5 = px.bar(stats, x='Post Type', y='mean', error_y='Margin',
              title='Average Engagement Rate by Post Type (with margin of error)')
fig5.update_layout(yaxis_title='Average Engagement Rate (%)', xaxis_title=None)
fig5.update_yaxes(range=[0, 22])
fig5.show()

display(stats.round(3))

,Post Type,mean,std,count,Margin
0,Image,18.43,7.15,12399,0.13
1,Reel,18.61,7.22,12599,0.13
2,Story,18.60,7.19,12517,0.13
3,Video,18.51,7.21,12485,0.13


In [67]:
long = format_summary.melt(id_vars='Post Type',
                           value_vars=['Avg_Likes', 'Avg_Comments', 'Avg_Shares', 'Avg_Saves'],
                           var_name='Interaction', value_name='Average per post')
long['Interaction'] = long['Interaction'].str.replace('Avg_', '')

fig6 = px.bar(long, x='Post Type', y='Average per post', facet_col='Interaction',
              text='Average per post',
              title='Average Likes, Comments, Shares and Saves per Post')

fig6.update_yaxes(matches=None, showticklabels=True, rangemode='tozero')
fig6.update_xaxes(title=None)
fig6.update_traces(texttemplate='%{text:,.0f}', textposition='outside', cliponaxis=False)
fig6.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig6.show()

**Insight, So what and Action:** Reel does not clearly win. Reel has the highest average engagement rate at 18.61%, but that is only 0.01 above Story (18.60%), 0.10 above Video (18.51%) and 0.18 above Image (18.43%). The margin-of-error lines on the bars overlap, and the p-value across the four formats is 0.152, well above 0.05, so the gaps could easily be luck. Average likes (about 2,950), comments (about 370), shares (about 162) and saves (about 178) per post are almost the same for every format. Even if the Reel lead over Image were real, it would mean only about 35 extra interactions per post. The so-what is that format alone does not explain why some posts do better. The action is not to move everything to Reels, but to choose the format that fits the message and to test it, for example by posting the same content as a Reel and as an Image and comparing engagement rate.

## Visual 6: Day of the Week

In [68]:
day_stats = (clean.groupby('Weekday', observed=True)['Engagement Rate']
             .agg(['mean', 'std', 'count']).reset_index())
day_stats['Margin'] = 1.96 * day_stats['std'] / np.sqrt(day_stats['count'])

overall = clean['Engagement Rate'].mean()

fig7 = px.bar(day_stats, x='Weekday', y='mean', error_y='Margin',
              title='Average Engagement Rate by Day of the Week')
fig7.add_hline(y=overall, line_dash='dash',
               annotation_text=f'Overall average {overall:.2f}%')
fig7.update_yaxes(range=[0, 22], title='Average Engagement Rate (%)')
fig7.update_xaxes(title=None)
fig7.show()

display(day_stats.round(3))

,Weekday,mean,std,count,Margin
0,Monday,18.48,7.22,7094,0.17
1,Tuesday,18.57,7.22,7355,0.17
2,Wednesday,18.63,7.17,7179,0.17
3,Thursday,18.48,7.16,7068,0.17
4,Friday,18.57,7.16,7245,0.17
5,Saturday,18.59,7.18,7031,0.17
6,Sunday,18.42,7.26,7028,0.17


In [69]:
posts_by_day = (clean.groupby('Weekday', observed=True).size()
                .reset_index(name='Posts'))

fig8 = px.bar(posts_by_day, x='Weekday', y='Posts', text='Posts',
              title='Number of Posts by Day of the Week')
fig8.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)
fig8.update_yaxes(rangemode='tozero')
fig8.update_xaxes(title=None)
fig8.show()

In [70]:
from scipy import stats

def luck_test(column):
    groups = [g['Engagement Rate'] for _, g in clean.groupby(column, observed=True)]
    return round(stats.f_oneway(*groups).pvalue, 3)

print("Post Type p-value:", luck_test('Post Type'))
print("Weekday p-value:  ", luck_test('Weekday'))

Post Type p-value: 0.152
Weekday p-value:   0.566


**Insight, So what and Action:** No day is clearly the best. Wednesday scored highest at 18.63% and Sunday lowest at 18.42%, a gap of only 0.21. Each bar has a margin of error of about 0.17, so the ranges overlap, and the p-value of 0.566 is well above 0.05, meaning gaps like these could easily happen by luck. All seven days sit within 0.1 of the overall average of 18.54%. The so-what is that posting day does not drive results here, so a "best day" rule would be a guess. The action is to post on any day, spend the effort on the content itself, and test a few days properly before making a rule.

## Visual 7: Engagement Funnel

In [71]:
import plotly.graph_objects as go

stages = ['Impressions', 'Reach', 'Total Engagement', 'Profile Visits', 'Follows']
values = [clean['Impressions'].sum(), clean['Reach'].sum(), clean['Total Engagement'].sum(),
          clean['Profile Visits'].sum(), clean['Follows'].sum()]

fig9 = go.Figure(go.Funnel(y=stages, x=values, textinfo='value+percent previous'))
fig9.update_layout(title='Engagement Funnel: from Views to Follows (drop-off view)')
fig9.show()

In [72]:
funnel = pd.DataFrame({'Stage': stages, 'Total': values})
funnel['% of previous stage'] = (funnel['Total'] / funnel['Total'].shift(1) * 100).round(1)
funnel['% of impressions']    = (funnel['Total'] / funnel['Total'].iloc[0] * 100).round(2)
display(funnel)

,Stage,Total,% of previous stage,% of impressions
0,Impressions,1276045750,NaN,100.00
1,Reach,987423288,77.40,77.38
2,Total Engagement,183164417,18.50,14.35
3,Profile Visits,29634798,16.20,2.32
4,Follows,8866710,29.90,0.69


In [73]:
mix = clean[['Likes', 'Comments', 'Shares', 'Saves']].sum()
mix_pct = (mix / mix.sum() * 100).round(1)

fig10 = px.bar(x=mix_pct.index, y=mix_pct.values, text=mix_pct.values,
               title='What People Do: Share of All Interactions (%)')
fig10.update_traces(texttemplate='%{text}%', textposition='outside', cliponaxis=False)
fig10.update_layout(xaxis_title=None, yaxis_title='% of all interactions')
fig10.update_yaxes(range=[0, 100])
fig10.show()

**Insight, So what and Action:** Across all posts, about 77% of impressions reached a different account (987M of 1,276M), and only about 18.5% of that reach turned into an interaction (183M). Of all interactions, 80.7% are likes, while comments are 10.1%, saves 4.8% and shares 4.4%. Profile visits are only 3% of reach (29.6M), and about 30% of profile visitors follow, so only about 0.7% of impressions end in a follow. The biggest drop is between interactions and profile visits, where only about 16% carry on. This is a drop-off view, not a proven journey, because the steps count different things and we cannot show that one step happens after another. The so-what is that the brand gets plenty of likes, which are quick and easy, but few of the stronger actions such as saves and shares, and few people go on to visit or follow the profile. The action is to make the next step easier and test it: add clear calls to action such as "save this for later" and "tap our profile," create content people want to save or share, and compare the profile visit rate before and after.

## Visual 8: Hashtags

In [74]:
h_stats = (hashtag_df.groupby('Hashtag')['Engagement Rate']
           .agg(['mean', 'std', 'count']).reset_index())
h_stats['Margin'] = 1.96 * h_stats['std'] / np.sqrt(h_stats['count'])

fig11 = px.bar(h_stats.sort_values('mean'), x='mean', y='Hashtag', orientation='h',
               error_x='Margin', title='Average Engagement Rate by Hashtag')
fig11.add_vline(x=clean['Engagement Rate'].mean(), line_dash='dash')
fig11.update_xaxes(range=[0, 22], title='Average Engagement Rate (%)')
fig11.update_yaxes(title=None)
fig11.show()
display(h_stats.round(3))

,Hashtag,mean,std,count,Margin
0,#BoldLooks,18.58,7.20,19956,0.10
1,#CrueltyFree,18.54,7.19,20197,0.10
2,#MakeupGoals,18.51,7.20,20018,0.10
3,#SugarCosmetics,18.52,7.21,19891,0.10
4,#VeganBeauty,18.61,7.20,19982,0.10


In [75]:
from scipy import stats

count_stats = clean.groupby('Hashtag Count')['Engagement Rate'].agg(['mean', 'count']).round(2).reset_index()
display(count_stats)
print("p-value:", round(stats.f_oneway(*[g['Engagement Rate'] for _, g in clean.groupby('Hashtag Count')]).pvalue, 3))

,Hashtag Count,mean,count
0,1,18.44,16625
1,2,18.63,16706
2,3,18.54,16669


p-value: 0.054


**Insight, So what and Action:** All five hashtags perform almost the same. #VeganBeauty is highest at 18.61% and #MakeupGoals is lowest at 18.51%, a gap of only 0.10, which is about the same size as each bar's margin of error (±0.10), so the ranges overlap. Posts that used 2 hashtags averaged 18.63%, against 18.44% for 1 hashtag and 18.54% for 3, and the p-value of 0.054 is just above the 0.05 line, so we cannot say the number of hashtags matters. The so-what is that no hashtag in this data is a clear winner or a clear loser, so there is no "bad hashtag" the brand should drop. Because posts with several hashtags count under each of them, these groups overlap and the comparison is only a rough guide. The action is to keep using the brand's core hashtags, such as #VeganBeauty, #BoldLooks and #CrueltyFree, which also reflect its identity, and to run a simple test of 2 hashtags against 3 on similar posts before setting any rule.

## Visual 9: Campaign, Audience Emotion and Engagement Source

In [76]:
def compare(col):
    s = clean.groupby(col)['Engagement Rate'].agg(['mean', 'std', 'count']).reset_index()
    s['Margin'] = 1.96 * s['std'] / np.sqrt(s['count'])
    fig = px.bar(s.sort_values('mean'), x='mean', y=col, orientation='h', error_x='Margin',
                 title=f'Average Engagement Rate by {col}')
    fig.update_xaxes(range=[0, 22], title='Average Engagement Rate (%)')
    fig.update_yaxes(title=None)
    fig.show()
    display(s.round(3))
    print(col, 'p-value:', luck_test(col))

for c in ['Campaign Type', 'Audience Emotion', 'Engagement Source']:
    compare(c)

,Campaign Type,mean,std,count,Margin
0,Festive Offer,18.44,7.18,9990,0.14
1,Giveaway,18.57,7.14,9998,0.14
2,Influencer Collab,18.59,7.25,9903,0.14
3,Product Launch,18.49,7.17,9923,0.14
4,Tutorial,18.58,7.24,10186,0.14


Campaign Type p-value: 0.478


,Audience Emotion,mean,std,count,Margin
0,Confident,18.57,7.22,9891,0.14
1,Curious,18.62,7.23,10027,0.14
2,Excited,18.47,7.22,10061,0.14
3,Happy,18.50,7.14,9884,0.14
4,Inspired,18.51,7.16,10137,0.14


Audience Emotion p-value: 0.562


,Engagement Source,mean,std,count,Margin
0,Collab,18.59,7.21,12633,0.13
1,Influencer,18.58,7.18,12501,0.13
2,Organic,18.44,7.24,12508,0.13
3,Paid,18.53,7.15,12358,0.13


Engagement Source p-value: 0.349


In [77]:
display(clean.groupby('Engagement Source')[['Reach', 'Impressions', 'Profile Visits', 'Follows']].mean().round(0))

,Reach,Impressions,Profile Visits,Follows
Engagement Source,,,,
Collab,"19,707.00","25,452.00",593.00,177.00
Influencer,"19,699.00","25,444.00",589.00,175.00
Organic,"19,970.00","25,835.00",600.00,179.00
Paid,"19,617.00","25,352.00",589.00,178.00


**Insight, So what and Action:** Campaign type, audience emotion and engagement source all perform almost the same. Influencer Collab is the highest campaign at 18.60% and Festive Offer the lowest at 18.44%; Curious is the highest emotion at 18.62% and Excited the lowest at 18.47%; and Collab is the highest source at 18.59% against 18.44% for Organic. Every gap is about 0.15, which is the same size as the margin of error, and all three p-values (0.478, 0.562 and 0.349) are far above 0.05, so these differences could easily be luck. Paid posts also did not reach more people than Organic ones (about 19,600 against 19,970 on average), although the data has no budget information, so we cannot judge the return on spending. The so-what is that no campaign, emotion or source clearly explains why some posts do better, and paying for posts shows no extra reach in this data. The action is to avoid shifting budget toward any one campaign or paid promotion based on these results, and instead to run small tests, such as the same content with and without paid support, and compare reach and engagement rate.

## Visual 10: Content Recipe Lab (unique feature)

The Recipe Lab tests 400 content "recipes" (post type × campaign × audience emotion × engagement source). Each recipe is compared with the overall average engagement rate. A recipe only gets a **DO** or **DON'T** verdict if it passes a strict statistical bar. Otherwise it is **TEST**, which means "we cannot tell yet."

In [78]:
from scipy import stats

RECIPE_COLS = ['Post Type', 'Campaign Type', 'Audience Emotion', 'Engagement Source']
overall_mean = clean['Engagement Rate'].mean()
overall_std  = clean['Engagement Rate'].std()

N_RECIPES = clean.groupby(RECIPE_COLS).ngroups
Z_STRICT  = stats.norm.ppf(1 - 0.05 / (2 * N_RECIPES))
Z_HINT    = 1.96
MIN_POSTS = 30

def score(sub):
    n    = len(sub)
    avg  = sub['Engagement Rate'].mean()
    lift = avg - overall_mean
    se   = overall_std / np.sqrt(n)
    return n, avg, lift, 1.96 * se, lift / se

def verdict(n, z):
    if n < MIN_POSTS:  return '⚪ TOO FEW POSTS'
    if z >= Z_STRICT:  return '🟢 DO'
    if z <= -Z_STRICT: return "🔴 DON'T"
    if z >= Z_HINT:    return '🟡 TEST (leans up)'
    if z <= -Z_HINT:   return '🟡 TEST (leans down)'
    return '🟡 TEST'

rows = []
for keys, sub in clean.groupby(RECIPE_COLS):
    n, avg, lift, margin, z = score(sub)
    rows.append(dict(zip(RECIPE_COLS, keys)) |
                {'Posts': n, 'Avg_ER': avg, 'Lift': lift, 'Margin': margin, 'z': z, 'Verdict': verdict(n, z)})
recipes = pd.DataFrame(rows)

print("Recipes:", N_RECIPES, "| strict bar (z):", round(Z_STRICT, 2))
print(recipes['Posts'].describe().round(0))

Recipes: 400 | strict bar (z): 3.84
count   400.00
mean    125.00
std      11.00
min      92.00
25%     118.00
50%     125.00
75%     133.00
max     156.00
Name: Posts, dtype: float64


In [79]:
show = RECIPE_COLS + ['Posts', 'Avg_ER', 'Lift', 'z', 'Verdict']
display(recipes['Verdict'].value_counts())
display(recipes.sort_values('z', ascending=False).head(5)[show].round(2))
display(recipes.sort_values('z').head(5)[show].round(2))

codes   = clean.groupby(RECIPE_COLS).ngroup().values
K       = codes.max() + 1
n_posts = np.bincount(codes, minlength=K)
er      = clean['Engagement Rate'].values
rng     = np.random.default_rng(42)

def z_scores(values):
    avg = np.bincount(codes, weights=values, minlength=K) / n_posts
    return (avg - overall_mean) / (overall_std / np.sqrt(n_posts))

real_leans     = (np.abs(z_scores(er)) >= Z_HINT).sum()
shuffled_leans = [(np.abs(z_scores(rng.permutation(er))) >= Z_HINT).sum() for _ in range(1000)]
print("Recipes that 'lean' in the real data:", real_leans)
print("Average in 1,000 randomly shuffled versions:", round(np.mean(shuffled_leans), 1))

,count
Verdict,
🟡 TEST,376
🟡 TEST (leans down),12
🟡 TEST (leans up),12


,Post Type,Campaign Type,Audience Emotion,Engagement Source,Posts,Avg_ER,Lift,z,Verdict
262,Story,Product Launch,Confident,Organic,124,20.96,2.42,3.75,🟡 TEST (leans up)
184,Reel,Tutorial,Curious,Collab,136,20.41,1.87,3.03,🟡 TEST (leans up)
224,Story,Giveaway,Curious,Collab,141,20.30,1.77,2.92,🟡 TEST (leans up)
284,Story,Tutorial,Curious,Collab,131,20.21,1.68,2.66,🟡 TEST (leans up)
169,Reel,Product Launch,Excited,Influencer,133,20.07,1.53,2.46,🟡 TEST (leans up)


,Post Type,Campaign Type,Audience Emotion,Engagement Source,Posts,Avg_ER,Lift,z,Verdict
33,Image,Giveaway,Happy,Influencer,121,16.68,-1.85,-2.83,🟡 TEST (leans down)
74,Image,Product Launch,Happy,Organic,112,16.62,-1.91,-2.81,🟡 TEST (leans down)
98,Image,Tutorial,Inspired,Organic,137,16.83,-1.70,-2.77,🟡 TEST (leans down)
267,Story,Product Launch,Curious,Paid,105,16.73,-1.81,-2.57,🟡 TEST (leans down)
266,Story,Product Launch,Curious,Organic,115,16.85,-1.68,-2.51,🟡 TEST (leans down)


Recipes that 'lean' in the real data: 24
Average in 1,000 randomly shuffled versions: 19.6


In [80]:
def recipe_lookup(post_type='Any', campaign='Any', emotion='Any', source='Any'):
    picks = {'Post Type': post_type, 'Campaign Type': campaign,
             'Audience Emotion': emotion, 'Engagement Source': source}
    sub = clean
    for col, val in picks.items():
        if val != 'Any':
            sub = sub[sub[col] == val]
    n, avg, lift, margin, z = score(sub)
    return {'Posts': n, 'Avg rate': round(avg, 2), 'Lift': round(lift, 2),
            'Margin': round(margin, 2), 'z': round(z, 2), 'Verdict': verdict(n, z)}

def posts_needed(lift_pp, alpha=0.05, power=0.8):
    z_a, z_b = stats.norm.ppf(1 - alpha / 2), stats.norm.ppf(power)
    return int(np.ceil(2 * ((z_a + z_b) * overall_std / lift_pp) ** 2))

print(recipe_lookup(post_type='Reel', campaign='Tutorial'))
print(recipe_lookup('Story', 'Product Launch', 'Confident', 'Organic'))
for d in [0.2, 0.5, 1, 2]:
    print(f"To confirm a {d}-point difference: {posts_needed(d):,} posts per group")

{'Posts': 2533, 'Avg rate': np.float64(18.5), 'Lift': np.float64(-0.04), 'Margin': np.float64(0.28), 'z': np.float64(-0.27), 'Verdict': '🟡 TEST'}
{'Posts': 124, 'Avg rate': np.float64(20.96), 'Lift': np.float64(2.42), 'Margin': np.float64(1.27), 'z': np.float64(3.75), 'Verdict': '🟡 TEST (leans up)'}
To confirm a 0.2-point difference: 20,321 posts per group
To confirm a 0.5-point difference: 3,252 posts per group
To confirm a 1-point difference: 813 posts per group
To confirm a 2-point difference: 204 posts per group


**Insight, So what and Action:** The Recipe Lab compares 400 content recipes, each with 92 to 156 posts, against the overall average engagement rate of 18.54%. No recipe passes the strict bar needed to call it a proven winner or loser (a z-score of 3.84, raised because checking 400 recipes makes some look special by luck). The best recipe, a Story for a Product Launch with a Confident tone from an Organic source, reaches 20.96% (z = 3.75), and the weakest, an Image Giveaway with a Happy tone from an Influencer source, falls to 16.68%, but both are only hints. A shuffle test shows that random data produces about 20 "leaning" recipes on average, and the real data has 24, so these leans look like normal luck. The so-what is that in this dataset the brand cannot honestly say which content to make or avoid, and the Recipe Lab says so instead of inventing rules. The action is to treat the top leaning recipes as ideas to test: run an A/B test with about 800 posts per group to confirm a 1-point difference, and re-run the Lab on real, larger data, where the same engine can flag true Do and Don't recipes.

## Final Export

In [81]:
FINAL_PATH = 'sugar_instagram_clean.csv'
clean.to_csv(FINAL_PATH, index=False)

check = pd.read_csv(FINAL_PATH, parse_dates=['Post Date'])
assert check.shape == clean.shape
print("Saved:", check.shape)

Saved: (50000, 25)


**Summary of data preparation:** The raw file (50,000 rows, 14 columns) had no missing values, duplicates or invalid values. High-performing posts flagged as outliers were kept because they are valid. Eleven calculated fields were added, giving a final file of 50,000 rows and 25 columns (`sugar_instagram_clean.csv`), which the Streamlit app loads.

**Limitations and assumptions:**
- The dataset has no caption, post ID or time of day, so a post ID was created from the row order and we cannot explain why a post worked.
- The data looks synthetic: every category is almost perfectly balanced and engagement rates are spread evenly.
- Engagement rate is a project-defined measure: (Likes + Comments + Shares + Saves) ÷ Reach × 100.
- "Total reach" adds each post's reach, so the same person can be counted many times.
- April 2024 and April 2025 are partial months.
- Results show association, not cause. This is descriptive analytics, and no predictive model is used.